# ETL-22 · dim_profissao e dim_local_obito

- **Etapa:** ETL-22, dim_profissao e dim_local_obito
- **Responsável:** Davi Mello
- **Issue:** ETL-22
- **Entradas:** `data/intermediarios/06_limpo.parquet` (profissão, ETL-17) e `data/intermediarios/05_nulos_categorias.parquet` (local do óbito, ETL-16)
- **Saídas:** `data/intermediarios/dims/dim_profissao.parquet` e `dim_local_obito.parquet`

Cria as duas últimas dimensões descritivas do modelo estrela, cada uma com chave substituta e os membros especiais -1 (NÃO INFORMADO) e -2 (NÃO SE APLICA).

## 1. Setup

In [1]:
from pathlib import Path
import pandas as pd

# Raiz do repositório: sobe de pasta em pasta até achar a pasta .git
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())

INTERM = RAIZ / "data" / "intermediarios"      # .parquet (o .gitignore já ignora)
DIMS   = INTERM / "dims"                       # saída: uma pasta só para as dimensões
DIMS.mkdir(parents=True, exist_ok=True)

# Na execução final as duas variáveis apontam para o mesmo arquivo (a saída da etapa anterior)
ENTRADA_PROFISSAO = INTERM / "06_limpo.parquet"           # saída da ETL-17: profissao e grupo_profissao
ENTRADA_LOCAL     = INTERM / "05_nulos_categorias.parquet"  # saída da ETL-16: local_obito e flag_obito

## 2. Carregar a entrada

In [2]:
# lê só as colunas que importam (os parquets têm 34)
df_prof  = pd.read_parquet(ENTRADA_PROFISSAO, columns=['id_pessoa', 'profissao', 'grupo_profissao'])
df_local = pd.read_parquet(ENTRADA_LOCAL, columns=['id_pessoa', 'local_obito', 'flag_obito'])
print('profissão:', df_prof.shape, '| local do óbito:', df_local.shape)

# os dois arquivos têm que ser da mesma base: mesmas pessoas
assert len(df_prof) == len(df_local) and df_prof['id_pessoa'].is_unique
assert df_prof['id_pessoa'].isin(df_local['id_pessoa']).all()
print('mesmas', len(df_prof), 'pessoas nos dois arquivos')

profissão: (806977, 3) | local do óbito: (806977, 3)
mesmas 806977 pessoas nos dois arquivos


## 3. Diagnóstico ANTES

In [3]:
def diagnostico(df: pd.DataFrame) -> pd.DataFrame:
    """Tabela-resumo usada antes e depois: tipo, % de nulos e nº de valores distintos por coluna."""
    return pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'pct_nulo': (df.isna().mean() * 100).round(2),
        'n_distintos': df.nunique(),
    })

antes = pd.concat([
    diagnostico(df_prof[['profissao', 'grupo_profissao']]),
    diagnostico(df_local[['local_obito']]),
])
antes

,dtype,pct_nulo,n_distintos
profissao,str,10.64,561
grupo_profissao,str,0.00,13
local_obito,str,0.00,5


## 4. dim_local_obito

O local do óbito vem da ETL-16: quem não morreu está como `NÃO SE APLICA` e o óbito sem local informado como `NÃO INFORMADO`. Sobram só os três locais reais, que viram as linhas da dimensão.

In [4]:
# como o local do óbito chega da ETL-16
print(df_local['local_obito'].value_counts(dropna=False))

# NÃO SE APLICA tem que ser exatamente quem não morreu
nao_se_aplica = df_local['local_obito'] == 'NÃO SE APLICA'
assert (nao_se_aplica == ~df_local['flag_obito']).all()
print('\nóbitos (flag_obito):', int(df_local['flag_obito'].sum()))

local_obito
NÃO SE APLICA               791184
ESTABELECIMENTO DE SAUDE      7940
VIA                           7214
OUTROS                         534
NÃO INFORMADO                  105
Name: count, dtype: int64

óbitos (flag_obito): 15793


In [5]:
LOCAIS = ['ESTABELECIMENTO DE SAUDE', 'OUTROS', 'VIA']

def criar_dim_local_obito(local: pd.Series) -> pd.DataFrame:
    """Uma linha por local de óbito, com chave sk_ e os membros -1 e -2."""
    valores = sorted(local[~local.isin(['NÃO INFORMADO', 'NÃO SE APLICA'])].unique())

    # só aceita os locais conhecidos, para nunca criar uma linha de lixo na dimensão
    assert set(valores) <= set(LOCAIS), f'local de óbito novo: {valores}'

    dim = pd.DataFrame({'sk_local_obito': range(1, len(valores) + 1), 'local_obito': valores})
    especiais = pd.DataFrame({
        'sk_local_obito': [-1, -2],
        'local_obito': ['NÃO INFORMADO', 'NÃO SE APLICA'],
    })
    return pd.concat([especiais, dim], ignore_index=True)

dim_local_obito = criar_dim_local_obito(df_local['local_obito'])
dim_local_obito

,sk_local_obito,local_obito
0,-1,NÃO INFORMADO
1,-2,NÃO SE APLICA
2,1,ESTABELECIMENTO DE SAUDE
3,2,OUTROS
4,3,VIA


## 5. dim_profissao

A dimensão tem uma linha por profissão já padronizada na ETL-17, com o `grupo_profissao`. Antes de criar, falta um cuidado com os nulos.

### 5.1 Nulos escondidos na profissão

A ETL-17 padronizou a escrita, mas duas grafias de "não informado" da própria fonte continuam como se fossem profissões e caíram no grupo `OUTROS`. Aqui só consulta: nada é alterado.

In [6]:
# grafias que dizem "não informado" ou "sem profissão" (só consulta)
contagem = df_prof['profissao'].value_counts()
suspeitas = contagem[contagem.index.str.contains('INFORMAD|DEFINID', na=False)]
resumo = suspeitas.to_frame('pessoas')
resumo['grupo_profissao'] = [df_prof.loc[df_prof['profissao'] == p, 'grupo_profissao'].iloc[0] for p in resumo.index]
resumo

,pessoas,grupo_profissao
profissao,,
NAO INFORMADA,636,OUTROS
NAO INFORMADO,298,OUTROS
SEM PROFISSAO DEFINIDA,9,OUTROS


### 5.2 Função que leva os nulos ao membro -1

`NAO INFORMADA` e `NAO INFORMADO` significam o mesmo que o campo vazio. Eles, o vazio e o `NÃO INFORMADO` que a ETL-16 grava passam a usar o membro -1. `SEM PROFISSAO DEFINIDA` (9 pessoas) é uma resposta e continua como profissão.

In [7]:
# grafias de "não informado" que a ETL-17 ainda deixa como profissão
NAO_INFORMADO = ['NAO INFORMADA', 'NAO INFORMADO', 'NÃO INFORMADO']

def profissao_para_dimensao(profissao: pd.Series) -> pd.Series:
    """Nulo e as grafias de "não informado" viram nulo: todos usam o membro -1 da dimensão."""
    return profissao.mask(profissao.isin(NAO_INFORMADO))

# a função não muda nada em quem já tem profissão
chave = profissao_para_dimensao(df_prof['profissao'])
print('nulos antes:', int(df_prof['profissao'].isna().sum()), '| depois:', int(chave.isna().sum()))

nulos antes: 85900 | depois: 86834


### 5.3 Criar a dimensão

In [8]:
def criar_dim_profissao(df: pd.DataFrame) -> pd.DataFrame:
    """Uma linha por profissão padronizada, com seu grupo, chave sk_ e os membros -1 e -2."""
    base = df[['profissao', 'grupo_profissao']].copy()
    base['profissao'] = profissao_para_dimensao(base['profissao'])

    # quem está sem profissão não vira linha: usa o membro -1
    base = base.dropna(subset=['profissao'])
    dim = base.drop_duplicates().sort_values('profissao').reset_index(drop=True)

    # cada profissão tem que ter um grupo só (senão a mesma chave teria dois grupos)
    assert dim['profissao'].is_unique, 'profissão com mais de um grupo'

    dim.insert(0, 'sk_profissao', range(1, len(dim) + 1))
    especiais = pd.DataFrame({
        'sk_profissao': [-1, -2],
        'profissao': ['NÃO INFORMADO', 'NÃO SE APLICA'],
        'grupo_profissao': ['NÃO INFORMADO', 'NÃO SE APLICA'],
    })
    return pd.concat([especiais, dim], ignore_index=True)

dim_profissao = criar_dim_profissao(df_prof)
print(len(dim_profissao), 'linhas')
dim_profissao.head(8)

561 linhas


,sk_profissao,profissao,grupo_profissao
0,-1,NÃO INFORMADO,NÃO INFORMADO
1,-2,NÃO SE APLICA,NÃO SE APLICA
2,1,ACOUGUEIRO,SERVICOS PESSOAIS E DOMESTICOS
3,2,ADESTRADOR DE ANIMAIS,OUTROS
4,3,ADMINISTRADOR,ADMINISTRATIVO
5,4,ADMINISTRADOR DE EMPRESAS,ADMINISTRATIVO
6,5,ADMINISTRADOR DE FAZENDA,ADMINISTRATIVO
7,6,ADMINISTRADOR RURAL,ADMINISTRATIVO


In [9]:
# quantas profissões tem cada grupo (12 grupos)
dim_profissao[dim_profissao['sk_profissao'] > 0]['grupo_profissao'].value_counts()

grupo_profissao
OUTROS                            305
INDUSTRIA E CONSTRUCAO             51
ADMINISTRATIVO                     50
TECNICOS E NIVEL SUPERIOR          40
SEGURANCA E SERVICO PUBLICO        23
SAUDE E EDUCACAO                   22
SERVICOS PESSOAIS E DOMESTICOS     18
TRANSPORTE E ENTREGA               17
COMERCIO E VENDAS                  17
AGRICULTURA E CAMPO                 6
SEM OCUPACAO                        6
AUTONOMO E EMPRESARIO               4
Name: count, dtype: int64

## 6. Conferir e salvar

Quatro conferências: a chave não se repete, nenhum nome passa do tamanho previsto no modelo, toda pessoa acha uma chave e os membros especiais têm o tamanho esperado.

In [10]:
def atribuir_sk_profissao(df: pd.DataFrame, dim: pd.DataFrame) -> pd.Series:
    """Chave de cada pessoa na dim_profissao (-1 quando não há profissão). A ETL-23 pode reutilizar."""
    mapa = dim[dim['sk_profissao'] > 0].set_index('profissao')['sk_profissao']
    return profissao_para_dimensao(df['profissao']).map(mapa).fillna(-1).astype('int64')

def atribuir_sk_local_obito(df: pd.DataFrame, dim: pd.DataFrame) -> pd.Series:
    """Chave de cada pessoa na dim_local_obito. A ETL-23 pode reutilizar."""
    mapa = dim.set_index('local_obito')['sk_local_obito']
    return df['local_obito'].map(mapa).astype('int64')

tabelas = [
    ('dim_profissao', dim_profissao, 'sk_profissao', ['profissao'], {'profissao': 80, 'grupo_profissao': 40}),
    ('dim_local_obito', dim_local_obito, 'sk_local_obito', ['local_obito'], {'local_obito': 30}),
]
for nome, dim, sk, natural, limites in tabelas:
    print(nome, '|', len(dim), 'linhas | sk única:', dim[sk].is_unique,
          '| nomes repetidos:', int(dim.duplicated(subset=natural).sum()))
    for col, limite in limites.items():
        assert dim[col].str.len().max() <= limite, f'{col} maior que {limite}'
    dim.to_parquet(DIMS / f'{nome}.parquet', index=False)

dim_profissao | 561 linhas | sk única: True | nomes repetidos: 0
dim_local_obito | 5 linhas | sk única: True | nomes repetidos: 0


In [11]:
# cada pessoa tem que achar uma chave nas duas dimensões
df_prof['sk_profissao'] = atribuir_sk_profissao(df_prof, dim_profissao)
df_local['sk_local_obito'] = atribuir_sk_local_obito(df_local, dim_local_obito)

pessoas_prof = df_prof['sk_profissao'].value_counts()
pessoas_local = df_local['sk_local_obito'].value_counts()
assert pessoas_prof.sum() == len(df_prof) and pessoas_local.sum() == len(df_local)
print('pessoas com chave | profissão:', int(pessoas_prof.sum()), '| local do óbito:', int(pessoas_local.sum()))

# o grupo da pessoa (ETL-17) é o mesmo da dimensão, menos nas que foram para -1
com_chave = df_prof[df_prof['sk_profissao'] > 0].merge(dim_profissao, on='sk_profissao', suffixes=('', '_dim'))
print('grupos diferentes da dimensão:', int((com_chave['grupo_profissao'] != com_chave['grupo_profissao_dim']).sum()))

pessoas com chave | profissão: 806977 | local do óbito: 806977
grupos diferentes da dimensão: 0


In [12]:
# membros especiais: quantas pessoas usam cada um
depois = pd.DataFrame({
    'linhas': [len(dim_profissao), len(dim_local_obito)],
    'pessoas_-1 (NÃO INFORMADO)': [int(pessoas_prof.get(-1, 0)), int(pessoas_local.get(-1, 0))],
    'pessoas_-2 (NÃO SE APLICA)': [int(pessoas_prof.get(-2, 0)), int(pessoas_local.get(-2, 0))],
}, index=['dim_profissao', 'dim_local_obito'])
depois

,linhas,pessoas_-1 (NÃO INFORMADO),pessoas_-2 (NÃO SE APLICA)
dim_profissao,561,86834,0
dim_local_obito,5,105,791184


In [13]:
# local do óbito: pessoas por linha da dimensão (-2 = quem não morreu)
dim_local_obito.assign(pessoas=dim_local_obito['sk_local_obito'].map(pessoas_local))

,sk_local_obito,local_obito,pessoas
0,-1,NÃO INFORMADO,105
1,-2,NÃO SE APLICA,791184
2,1,ESTABELECIMENTO DE SAUDE,7940
3,2,OUTROS,534
4,3,VIA,7214


In [14]:
# profissão: de onde vêm as pessoas do membro -1 (grupo que a ETL-17 tinha dado a elas)
df_prof[df_prof['sk_profissao'] == -1]['grupo_profissao'].value_counts()

grupo_profissao
NÃO INFORMADO    85900
OUTROS             934
Name: count, dtype: int64

## 7. Resumo

**O que foi feito:** criação das duas últimas dimensões descritivas do modelo estrela, cada uma com chave substituta `sk_` e os membros especiais -1 (NÃO INFORMADO) e -2 (NÃO SE APLICA). Os arquivos foram salvos em `data/intermediarios/dims/`.

**Resultado:**
- `dim_profissao`: 561 linhas (559 profissões padronizadas, cada uma com seu `grupo_profissao`, mais os 2 membros especiais).
- `dim_local_obito`: 5 linhas (ESTABELECIMENTO DE SAUDE, OUTROS e VIA, mais os 2 membros especiais).

**Nulos escondidos:** `NAO INFORMADA` (636) e `NAO INFORMADO` (298) são "não informado" escrito como profissão. A ETL-17 os deixou no grupo `OUTROS`. Aqui eles usam o membro -1, que passou a ter 86.834 pessoas (85.900 campos vazios e 934 dessas grafias). `SEM PROFISSAO DEFINIDA` (9 pessoas) continua como profissão.

**Local do óbito:** 791.184 pessoas usam o -2 (não morreram) e 105 usam o -1 (morreram, sem local). Os 15.793 óbitos se dividem em 7.940 em estabelecimento de saúde, 7.214 na via e 534 em outros locais.

**O que foi conferido:** as chaves são únicas, nenhum nome se repete, cada profissão tem um grupo só e as 806.977 pessoas encontram uma chave nas duas dimensões. O membro -2 da `dim_profissao` não é usado (todos têm profissão ou não informada); ele existe só para seguir a regra de que toda dimensão tem os dois membros.

**Para a ETL-23 (tabela fato):** as funções `atribuir_sk_profissao` e `atribuir_sk_local_obito` mostram como ligar cada pessoa à chave. A profissão precisa passar por `profissao_para_dimensao` antes do join, porque a grafia `NAO INFORMADO` não está na dimensão.

**Observação:** o ideal é que a ETL-17 trate `NAO INFORMADA` e `NAO INFORMADO` como nulo. Se isso for feito lá, esta etapa continua valendo sem mudar, porque a função aceita as duas formas.